# GlyphBench Evaluation Results

Paper-ready figures for LLM agent performance across 249+ environments.

In [ ]:
import json
from pathlib import Path

import matplotlib.pyplot as plt
import matplotlib as mpl
import numpy as np
import pandas as pd
import seaborn as sns

# Paper-ready style
plt.rcParams.update({
    'figure.dpi': 150,
    'savefig.dpi': 300,
    'font.size': 10,
    'font.family': 'serif',
    'axes.titlesize': 11,
    'axes.labelsize': 10,
    'xtick.labelsize': 8,
    'ytick.labelsize': 8,
    'legend.fontsize': 8,
    'figure.figsize': (7, 4),
    'axes.grid': True,
    'grid.alpha': 0.3,
    'axes.spines.top': False,
    'axes.spines.right': False,
})

SUITE_ORDER = ['minigrid', 'minihack', 'atari', 'procgen', 'craftax', 'classics']
SUITE_COLORS = dict(zip(SUITE_ORDER, sns.color_palette('husl', len(SUITE_ORDER))))
FIGURES_DIR = Path('figures')
FIGURES_DIR.mkdir(exist_ok=True)

## Load Results

In [ ]:
RESULTS_DIR = Path('../cluster_manager/results')

def load_all_results(results_dir: Path) -> pd.DataFrame:
    """Load all per-env results across all models and harness modes."""
    rows = []
    for model_dir in sorted(results_dir.iterdir()):
        if not model_dir.is_dir():
            continue
        model_name = model_dir.name
        for harness_dir in sorted(model_dir.iterdir()):
            if not harness_dir.is_dir():
                continue
            harness = harness_dir.name
            per_env = harness_dir / 'per_env'
            if not per_env.exists():
                continue
            for f in sorted(per_env.glob('*.json')):
                d = json.loads(f.read_text())
                if 'error' in d:
                    continue
                env_id = d['env_id']
                suite = env_id.split('/')[1].split('-')[0] if '/' in env_id else 'unknown'
                rows.append({
                    'model': model_name,
                    'harness': harness,
                    'env_id': env_id,
                    'suite': suite,
                    'mean_return': d['mean_return'],
                    'std_return': d['std_return'],
                    'mean_length': d['mean_length'],
                    'parse_failure_rate': d.get('parse_failure_rate', 0),
                    'n_episodes': d['n_episodes'],
                    'total_input_tokens': d.get('total_input_tokens', 0),
                    'total_output_tokens': d.get('total_output_tokens', 0),
                })
    return pd.DataFrame(rows)

df = load_all_results(RESULTS_DIR)
print(f'Loaded {len(df)} result entries')
print(f'Models: {df["model"].nunique()}')
print(f'Harness modes: {sorted(df["harness"].unique())}')
print(f'Envs: {df["env_id"].nunique()}')
df.head()

In [ ]:
import sys; sys.path.insert(0, '..')
from eval.scoring import compute_glyphbench_scores, print_leaderboard, SUITES

BASELINE_PATH = Path('random_baseline.json')
SCORES_PATH = Path('../cluster_manager/results')

if BASELINE_PATH.exists() and SCORES_PATH.exists():
    scores = compute_glyphbench_scores(SCORES_PATH, BASELINE_PATH)
    print_leaderboard(scores)
    
    # Build normalized DataFrame
    norm_rows = scores['per_env']
    df_norm = pd.DataFrame(norm_rows)
    print(f'\nNormalized scores: {len(df_norm)} entries')
else:
    print('Run eval/random_baseline.py first to generate baseline, then run experiments.')
    df_norm = pd.DataFrame()

## Figure 1: Main Results — Mean Return by Model and Suite

Grouped bar chart: one group per suite, one bar per model. The headline figure.

In [ ]:
# Use the best harness mode (history_cot) for the main comparison
best_harness = 'history_cot'
df_main = df[df['harness'] == best_harness].copy()
if df_main.empty:
    # Fallback to whatever harness is available
    best_harness = df['harness'].value_counts().index[0]
    df_main = df[df['harness'] == best_harness].copy()

# Aggregate by model x suite
agg = df_main.groupby(['model', 'suite'])['mean_return'].mean().reset_index()

fig, ax = plt.subplots(figsize=(12, 5))
models = sorted(agg['model'].unique())
x = np.arange(len(SUITE_ORDER))
width = 0.8 / len(models)

for i, model in enumerate(models):
    model_data = agg[agg['model'] == model]
    vals = [model_data[model_data['suite'] == s]['mean_return'].values[0]
            if s in model_data['suite'].values else 0 for s in SUITE_ORDER]
    label = model.split('_')[-1] if '_' in model else model
    ax.bar(x + i * width - 0.4 + width/2, vals, width, label=label, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels([s.title() for s in SUITE_ORDER])
ax.set_ylabel('Mean Return')
ax.set_title(f'GlyphBench: Mean Return by Suite ({best_harness})')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False)
ax.axhline(0, color='gray', lw=0.5, zorder=1)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig1_main_results.pdf', bbox_inches='tight')
plt.savefig(FIGURES_DIR / 'fig1_main_results.png', bbox_inches='tight')
plt.show()

## Figure 2: Scaling Curves — Model Size vs Performance

In [ ]:
# Extract model size from name (approximate)
def extract_size_b(model_name: str) -> float:
    """Extract parameter count in billions from model name."""
    import re
    m = re.search(r'(\d+\.?\d*)B', model_name, re.IGNORECASE)
    if m:
        return float(m.group(1))
    return 0

df_main['size_b'] = df_main['model'].apply(extract_size_b)

# Aggregate: overall mean return per model
model_perf = df_main.groupby('model').agg(
    mean_return=('mean_return', 'mean'),
    size_b=('size_b', 'first'),
).reset_index().sort_values('size_b')

fig, ax = plt.subplots(figsize=(7, 4.5))

# Color by model family
families = {}
for m in model_perf['model']:
    if 'Qwen3.5' in m: families[m] = 'Qwen3.5'
    elif 'Qwen3' in m: families[m] = 'Qwen3'
    elif 'DeepSeek' in m: families[m] = 'DeepSeek-R1'
    elif 'gemma' in m.lower(): families[m] = 'Gemma'
    elif 'llama' in m.lower() or 'Llama' in m: families[m] = 'Llama'
    elif 'mistral' in m.lower() or 'Mistral' in m: families[m] = 'Mistral'
    else: families[m] = 'Other'

model_perf['family'] = model_perf['model'].map(families)
family_colors = dict(zip(sorted(set(families.values())), sns.color_palette('Set2', len(set(families.values())))))

for family, group in model_perf.groupby('family'):
    ax.scatter(group['size_b'], group['mean_return'], label=family,
              color=family_colors[family], s=80, zorder=3, edgecolors='white', linewidth=0.5)
    # Connect points within family
    g = group.sort_values('size_b')
    ax.plot(g['size_b'], g['mean_return'], color=family_colors[family], alpha=0.4, lw=1.5)

ax.set_xscale('log')
ax.set_xlabel('Model Size (B parameters)')
ax.set_ylabel('Mean Return (all envs)')
ax.set_title('Scaling: Model Size vs GlyphBench Performance')
ax.legend(frameon=False)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig2_scaling.pdf', bbox_inches='tight')
plt.show()

## Figure 3: Harness Ablation — Markov vs History, Zero-shot vs CoT

In [ ]:
# Aggregate by harness mode across all envs
harness_agg = df.groupby(['model', 'harness'])['mean_return'].mean().reset_index()
harness_pivot = harness_agg.pivot(index='model', columns='harness', values='mean_return')

fig, axes = plt.subplots(1, 2, figsize=(12, 5))

# Left: Markov vs History (zero-shot)
ax = axes[0]
if 'markov_zeroshot' in harness_pivot.columns and 'history_zeroshot' in harness_pivot.columns:
    ax.scatter(harness_pivot['markov_zeroshot'], harness_pivot['history_zeroshot'],
              s=60, zorder=3, edgecolors='white', linewidth=0.5)
    for idx, row in harness_pivot.iterrows():
        if 'markov_zeroshot' in row and 'history_zeroshot' in row:
            ax.annotate(idx.split('_')[-1][:8], (row['markov_zeroshot'], row['history_zeroshot']),
                       fontsize=6, ha='left', va='bottom')
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.3, lw=1)
ax.set_xlabel('Markov (single obs)')
ax.set_ylabel('History (N recent obs)')
ax.set_title('Zero-shot: Markov vs History')

# Right: Zero-shot vs CoT (history mode)
ax = axes[1]
if 'history_zeroshot' in harness_pivot.columns and 'history_cot' in harness_pivot.columns:
    ax.scatter(harness_pivot['history_zeroshot'], harness_pivot['history_cot'],
              s=60, zorder=3, edgecolors='white', linewidth=0.5)
    for idx, row in harness_pivot.iterrows():
        if 'history_zeroshot' in row and 'history_cot' in row:
            ax.annotate(idx.split('_')[-1][:8], (row['history_zeroshot'], row['history_cot']),
                       fontsize=6, ha='left', va='bottom')
    lims = [min(ax.get_xlim()[0], ax.get_ylim()[0]), max(ax.get_xlim()[1], ax.get_ylim()[1])]
    ax.plot(lims, lims, 'k--', alpha=0.3, lw=1)
ax.set_xlabel('Zero-shot')
ax.set_ylabel('Chain-of-Thought')
ax.set_title('History mode: Zero-shot vs CoT')

plt.suptitle('Harness Mode Ablation', fontsize=12, y=1.02)
plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig3_harness_ablation.pdf', bbox_inches='tight')
plt.show()

## Figure 4: History Length Ablation

In [ ]:
# Filter history modes and group by history_len (extracted from harness name or config)
# For now, plot harness modes as categories
harness_order = ['markov_zeroshot', 'markov_cot', 'history_zeroshot', 'history_cot']
available = [h for h in harness_order if h in df['harness'].unique()]

fig, ax = plt.subplots(figsize=(8, 5))

for model in sorted(df['model'].unique()):
    model_df = df[df['model'] == model]
    vals = [model_df[model_df['harness'] == h]['mean_return'].mean() for h in available]
    label = model.split('_')[-1] if '_' in model else model
    ax.plot(range(len(available)), vals, 'o-', label=label, markersize=6)

ax.set_xticks(range(len(available)))
ax.set_xticklabels([h.replace('_', '\n') for h in available], fontsize=8)
ax.set_ylabel('Mean Return')
ax.set_title('Effect of Harness Mode on Performance')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False, fontsize=7)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig4_harness_modes.pdf', bbox_inches='tight')
plt.show()

## Figure 5: Per-Environment Breakdown (heatmap)

In [ ]:
# Top 30 most discriminative envs (highest variance across models)
df_best = df[df['harness'] == best_harness]
env_var = df_best.groupby('env_id')['mean_return'].var().nlargest(30)
top_envs = env_var.index.tolist()

pivot = df_best[df_best['env_id'].isin(top_envs)].pivot_table(
    index='env_id', columns='model', values='mean_return'
)
# Shorten env names for display
pivot.index = [e.split('/')[-1] for e in pivot.index]
pivot.columns = [c.split('_')[-1][:12] for c in pivot.columns]

fig, ax = plt.subplots(figsize=(max(10, len(pivot.columns) * 1.2), max(8, len(pivot) * 0.35)))
sns.heatmap(pivot, annot=True, fmt='.1f', cmap='RdYlGn', center=0,
           linewidths=0.5, ax=ax, cbar_kws={'label': 'Mean Return'})
ax.set_title(f'Per-Environment Returns — Top 30 Most Discriminative ({best_harness})')
ax.set_ylabel('')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig5_env_heatmap.pdf', bbox_inches='tight')
plt.show()

## Figure 6: Parse Failure Rate by Model

In [ ]:
parse_agg = df.groupby(['model', 'suite'])['parse_failure_rate'].mean().reset_index()

fig, ax = plt.subplots(figsize=(10, 5))
models = sorted(parse_agg['model'].unique())
x = np.arange(len(SUITE_ORDER))
width = 0.8 / len(models)

for i, model in enumerate(models):
    model_data = parse_agg[parse_agg['model'] == model]
    vals = [model_data[model_data['suite'] == s]['parse_failure_rate'].values[0]
            if s in model_data['suite'].values else 0 for s in SUITE_ORDER]
    label = model.split('_')[-1][:12]
    ax.bar(x + i * width - 0.4 + width/2, vals, width, label=label, zorder=3)

ax.set_xticks(x)
ax.set_xticklabels([s.title() for s in SUITE_ORDER])
ax.set_ylabel('Parse Failure Rate')
ax.set_title('JSON Parse Failure Rate by Model and Suite')
ax.legend(bbox_to_anchor=(1.05, 1), loc='upper left', frameon=False, fontsize=7)

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig6_parse_failures.pdf', bbox_inches='tight')
plt.show()

## Figure 7: Token Efficiency — Return per 1K Output Tokens

In [ ]:
# Compute return per 1K tokens
df_eff = df_best.copy()
df_eff['tokens_per_ep'] = df_eff['total_output_tokens'] / df_eff['n_episodes']
df_eff['return_per_1k_tokens'] = df_eff['mean_return'] / (df_eff['tokens_per_ep'] / 1000 + 1e-6)

eff_agg = df_eff.groupby('model').agg(
    mean_return=('mean_return', 'mean'),
    mean_tokens=('tokens_per_ep', 'mean'),
).reset_index()

fig, ax = plt.subplots(figsize=(8, 5))
ax.scatter(eff_agg['mean_tokens'] / 1000, eff_agg['mean_return'],
          s=80, zorder=3, edgecolors='white', linewidth=0.5)
for _, row in eff_agg.iterrows():
    ax.annotate(row['model'].split('_')[-1][:10],
               (row['mean_tokens'] / 1000, row['mean_return']),
               fontsize=7, ha='left', va='bottom')

ax.set_xlabel('Mean Output Tokens per Episode (K)')
ax.set_ylabel('Mean Return')
ax.set_title('Token Efficiency: Return vs Output Tokens')

plt.tight_layout()
plt.savefig(FIGURES_DIR / 'fig7_token_efficiency.pdf', bbox_inches='tight')
plt.show()

## Summary Table

In [ ]:
# Overall leaderboard
leaderboard = df_best.groupby('model').agg(
    overall_return=('mean_return', 'mean'),
    overall_std=('mean_return', 'std'),
    envs_evaluated=('env_id', 'nunique'),
    mean_parse_fail=('parse_failure_rate', 'mean'),
).round(4).sort_values('overall_return', ascending=False)

# Per-suite returns
for suite in SUITE_ORDER:
    suite_df = df_best[df_best['suite'] == suite]
    suite_ret = suite_df.groupby('model')['mean_return'].mean()
    leaderboard[f'{suite}_return'] = suite_ret

print('=== GlyphBench Leaderboard ===')
print(leaderboard.to_string())

# Save as CSV
leaderboard.to_csv(FIGURES_DIR / 'leaderboard.csv')
print(f'\nSaved to {FIGURES_DIR / "leaderboard.csv"}')